In [ ]:
import clingo
import pandas as pd
import os

# ==============================================================================
# 1. DRUG DATABASE LOADING ("Drugs_1.xlsx")
# ==============================================================================
file_name = "Drugs_1.xlsx"
output_csv = "selected_drugs_result_x.csv"

try:
    df = pd.read_excel(file_name)
    print(f"File '{file_name}' loaded successfully!")
    print(f"-> Found {len(df)} drug-target associations.")
except FileNotFoundError:
    print(f"CRITICAL ERROR: File '{file_name}' not found.")
    df = pd.DataFrame()

# ==============================================================================
# 2. TARGET DEFINITION
# ==============================================================================
perturbation_targets = {
     'RELA': 1, 'RXRA': 1, 'ACTB': 0
}

# ==============================================================================
# 3. ASP SOLVER WITH PAIRING AND CSV EXPORT
# ==============================================================================

def solve_best_effort(targets, drug_df):
    if drug_df.empty:
        print("Interruption: Drug database is empty.")
        return

    print(f"\n--- STARTING PHARMACOLOGICAL OPTIMIZATION ---")
    
    asp_program = []

    # --- A. FACTS DEFINITION ---
    for gene, val in targets.items():
        asp_program.append(f"required(\"{gene}\", {val}).")

    drugs_set = set()
    for _, row in drug_df.iterrows():
        gene = str(row['Gene']).strip().replace(" ", "_").replace("-", "_")
        drug = str(row['Drug']).strip().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "").replace("[", "").replace("]", "").replace("+", "_plus_")
        try:
            val = int(row['Boolean_Value'])
            asp_program.append(f"drug_effect(\"{drug}\", \"{gene}\", {val}).")
            drugs_set.add(drug)
        except:
            continue

    for d in drugs_set:
        asp_program.append(f"is_drug(\"{d}\").")

    # --- B. UPDATED ASP LOGIC ---
    asp_program.append("""
    { chosen(D) : is_drug(D) }.

    hits(D, G) :- chosen(D), drug_effect(D, G, V), required(G, V).

    applied(G, V) :- chosen(D), drug_effect(D, G, V).
    is_covered(G) :- required(G, V), applied(G, V).
    missed(G) :- required(G, _), not is_covered(G).
    side_effect(G) :- applied(G, _), not required(G, _).

    :- chosen(D), drug_effect(D, G, V_drug), required(G, V_req), V_drug != V_req.

    :~ missed(G). [1@3, G]
    :~ chosen(D). [1@2, D]
    :~ side_effect(G). [1@1, G]

    #show chosen/1.
    #show hits/2.
    #show missed/1.
    #show side_effect/1.
    """)

    ctl = clingo.Control()
    ctl.add("base", [], "\n".join(asp_program))
    ctl.ground([("base", [])])

    print("Solving... (Searching for the optimal combination...)")
    
    best_model_atoms = None
    with ctl.solve(yield_=True) as handle:
        for m in handle:
            best_model_atoms = m.symbols(shown=True)

    if best_model_atoms:
        print("\n>>> OPTIMAL RESULT FOUND <<<")
        
        results_to_save = []
        selected_drugs = set()
        covered_mapping = {}
        missed_genes = []
        all_side_effects = []

        for atom in best_model_atoms:
            args = [str(a).strip('"') for a in atom.arguments]
            if atom.name == "chosen":
                selected_drugs.add(args[0].replace("_plus_", "+"))
            if atom.name == "hits":
                drug_name = args[0].replace("_plus_", "+")
                gene_name = args[1]
                covered_mapping[gene_name] = drug_name
            if atom.name == "missed":
                missed_genes.append(args[0])
            if atom.name == "side_effect":
                all_side_effects.append(args[0])

        print(f"\nDRUG -> TARGET MAPPING:")
        print("-" * 40)
        for gene, drug in covered_mapping.items():
            required_val = targets[gene]
            action = "ACTIVATES" if required_val == 1 else "INHIBITS"
            print(f"  * {drug:<20} --[{action}]--> {gene}")
            
            results_to_save.append({
                'Drug': drug,
                'Target_Gene': gene,
                'Action': action,
                'Target_Status': 'COVERED'
            })

        if missed_genes:
            print(f"\nUNCOVERED TARGETS:")
            for m in missed_genes:
                print(f"  - {m}")
                results_to_save.append({
                    'Drug': 'N/A',
                    'Target_Gene': m,
                    'Action': 'N/A',
                    'Target_Status': 'MISSING'
                })

        df_res = pd.DataFrame(results_to_save)
        df_res.to_csv(output_csv, index=False)
        
        print("-" * 40)
        print(f"Results saved to: {output_csv}")
        print(f"Total side effects (unrequested hit genes): {len(all_side_effects)}")
             
    else:
        print("No solution found.")

if __name__ == "__main__":
    solve_best_effort(perturbation_targets, df)
